# Matriz de grafos

In [3]:
# -*- coding: utf-8 -*-

from __future__ import annotations
from pathlib import Path
import os
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple

import math
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon, MultiPolygon
from shapely.ops import nearest_points, unary_union
from shapely.strtree import STRtree
from scipy.spatial import cKDTree
import networkx as nx
import argparse
import sys
import logging
import time

# ------------------------- Config -------------------------
@dataclass
class Config:
    CRS: str = "EPSG:31983"

    # Velocidade de caminhada (para weight em minutos)
    WALK_SPEED: float = 70.0  # m/min (~4.2 km/h)

    # Caminho base dos dados
    dataset_dir: Path = Path(
        os.environ.get(
            "DATA_DIR",
            "G:/.shortcut-targets-by-id/1MB59B0WEFqH12bJOuljBxyIcY9y6kCEr/Mapas - Seleção de Estudos de Caso/Banco de dados/teste_clusterizacao/cluster/dataset/isocronas",
        )
    ).expanduser()

    # Nomes padrão de arquivos de entrada e saída
    quadras_file: str = "quadras.gpkg"
    centroides_file: str = "centroide.gpkg"
    candidatos_file: str = "candidatos_hotspot.gpkg"   # << sementes quentes (1ª rodada)
    barreiras_file: str = "barreiras.gpkg"
    nos_out: str = "grafo_nos_queen.gpkg"
    arestas_out: str = "grafo_arestas_queen.gpkg"

    # Parâmetros do grafo perimetral + travessias
    PERIM_STEP_M: float = 30.0          # densidade do anel (pontos a cada X m)
    MAX_CROSS_GAP_M: float = 45.0       # vão máximo (rua) para criar travessia
    NEIGHBOR_SEARCH_BUFFER_M: float = 60.0  # buffer p/ buscar vizinhos

    # Barreiras (para ignorar travessias que cortem obstáculos grandes)
    B_LIM: float = 60.0                 # metragem mínima de intersecção para considerar "barreira"
    BARRIER_SECTION_MIN: float = 350.0  # dimensão mínima do obstáculo "grande"

CFG = Config()

# ------------------------- Log -------------------------
LOG_FMT = "%(levelname)s | %(asctime)s | %(name)s | %(message)s"
logging.basicConfig(level=logging.INFO, format=LOG_FMT, datefmt="%H:%M:%S")
logger = logging.getLogger("grafo_builder")

def _now():
    return time.strftime("%H:%M:%S")

# ------------------------- Utils de caminho -------------------------
def _file(name: str | Path) -> Path:
    """Resolve caminho relativo ao dataset_dir, se necessário."""
    p = Path(name).expanduser()
    return p if p.is_absolute() else (CFG.dataset_dir / p)

# ------------------------- Barreiras -------------------------
def _axes_lengths(geom) -> Tuple[float, float]:
    """
    Estima os eixos (menor, maior) de um polígono via retângulo mínimo girado.
    Usado para decidir se um obstáculo é "grande".
    """
    try:
        mrr = getattr(geom, "minimum_rotated_rectangle", None) or geom.envelope
        if mrr.is_empty:
            return 0.0, 0.0
        poly = mrr if getattr(mrr, "geom_type", "") == "Polygon" else geom.envelope
        if poly.is_empty:
            return 0.0, 0.0
        coords = list(poly.exterior.coords) if hasattr(poly, "exterior") else []
        if len(coords) < 5:
            return 0.0, 0.0
        d01 = Point(coords[0]).distance(Point(coords[1]))
        d12 = Point(coords[1]).distance(Point(coords[2]))
        minor, major = (d01, d12) if d01 <= d12 else (d12, d01)
        return float(minor), float(major)
    except Exception:
        return 0.0, 0.0

def clean_barriers(barreiras: gpd.GeoSeries, *, simplify_tol: float = 0.1) -> gpd.GeoSeries:
    g = barreiras.copy()
    g = g.buffer(0)
    if simplify_tol > 0:
        g = g.simplify(simplify_tol)
    return g

def _iter_hits(hits):
    """Normaliza retorno do STRtree.query para list/tuple/ndarray."""
    try:
        import numpy as _np
        if isinstance(hits, _np.ndarray):
            return hits.tolist()
    except Exception:
        pass
    if isinstance(hits, (list, tuple)):
        return list(hits)
    # shapely 2 pode retornar generator-like; tenta converter
    try:
        return list(hits)
    except Exception:
        return []

def cross_big_barrier(seg: LineString, tree: Optional[STRtree], barreiras: gpd.GeoSeries | gpd.GeoDataFrame) -> bool:
    """
    Retorna True se a linha 'seg' cruza uma barreira "grande".
    Considera interseção e tamanho do obstáculo.
    """
    if tree is None:
        return False

    try:
        hits = tree.query(seg, predicate="intersects")
    except Exception:
        hits = tree.query(seg)

    inter_len = 0.0
    for cand in _iter_hits(hits):
        geom = cand.geometry if hasattr(cand, "geometry") else cand
        if geom is None or geom.is_empty:
            continue
        minor, major = _axes_lengths(geom)
        # só conta interseção se o obstáculo for "grande"
        if (minor >= CFG.BARRIER_SECTION_MIN) and (major >= CFG.BARRIER_SECTION_MIN):
            try:
                inter_len += seg.intersection(geom).length
            except Exception:
                # se der erro de overlay, considere que cruza
                return True

    return inter_len >= CFG.B_LIM

# ------------------------- Helpers de geometria -------------------------
def _main_polygon(geom) -> Optional[Polygon]:
    """
    Retorna o maior Polygon:
      - se geom for Polygon -> ele mesmo
      - se for MultiPolygon -> maior por área
    """
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, Polygon):
        return geom
    if isinstance(geom, MultiPolygon):
        if len(geom.geoms) == 0:
            return None
        return max(geom.geoms, key=lambda g: g.area)
    # tenta consertar geometrias ruins
    try:
        g2 = geom.buffer(0)
        if isinstance(g2, Polygon):
            return g2
        if isinstance(g2, MultiPolygon) and len(g2.geoms) > 0:
            return max(g2.geoms, key=lambda g: g.area)
    except Exception:
        pass
    return None

def _outer_ring_line(geom) -> Optional[LineString]:
    """Extrai o anel externo como LineString do maior polígono válido."""
    poly = _main_polygon(geom)
    if poly is None or poly.is_empty:
        return None
    try:
        return LineString(poly.exterior.coords)
    except Exception:
        return None

def _densify_line(line: LineString, step: float) -> List[Point]:
    """Cria pontos ao longo da linha a cada 'step' metros (inclui extremos)."""
    L = float(line.length)
    if L <= 0:
        return [Point(line.coords[0])]
    dists = list(np.arange(0.0, L, max(step, 1e-6))) + [L]
    pts = [line.interpolate(d) for d in dists]
    # garante >= 2 pontos
    if len(pts) < 2 and L > 0:
        pts = [line.interpolate(0.0), line.interpolate(L)]
    return pts

def _nearest_idx(pt: Point, pts: List[Point]) -> int:
    """Índice do ponto mais próximo em 'pts'."""
    if not pts:
        return -1
    arr = np.asarray([(p.x, p.y) for p in pts], dtype=float)
    kd = cKDTree(arr)
    _, idx = kd.query([pt.x, pt.y], k=1)
    return int(idx)

# ------------------------- Construtor do grafo -------------------------
def build_graph_perimeter_rook_with_crossings(
    quadras: gpd.GeoDataFrame,
    centroides_df: gpd.GeoDataFrame,
    barreiras: gpd.GeoSeries | gpd.GeoDataFrame,
    *,
    perim_step_m: float = 30.0,
    max_cross_gap_m: float = 45.0,
    neighbor_search_buffer_m: float = 60.0,
) -> nx.Graph:
    """
    Gera um grafo com:
      - Anel perimetral por quadra (nós e arestas ao longo do contorno);
      - Conector centroide -> perímetro;
      - Travessias entre perímetros de quadras vizinhas (vão <= max_cross_gap_m);
      - Evita travessias que cruzem barreiras "grandes".
    Todas as arestas possuem: length_m, weight (min), w (s), geometry.
    """
    # Preparos
    bar_series = barreiras.geometry if hasattr(barreiras, "geometry") else barreiras
    if bar_series is not None and len(bar_series) > 0:
        bar_series = clean_barriers(bar_series)
        try:
            bar_tree = STRtree(bar_series)
        except Exception:
            bar_tree = None
    else:
        bar_tree = None

    G = nx.Graph()
    sidx = quadras.sindex

    # Armazena elementos do perímetro
    perim_nodes: Dict[str, List[str]] = {}
    perim_pts: Dict[str, List[Point]] = {}
    perim_ls: Dict[str, LineString] = {}

    # Índice rápido de centroides por quadra_id
    cent_map: Dict[str, Point] = {}
    if "quadra_id" in centroides_df.columns:
        # pega o primeiro por quadra_id (se houver duplicados)
        firsts = centroides_df.drop_duplicates("quadra_id")
        cent_map = dict(zip(firsts["quadra_id"].astype(str), firsts.geometry))

    # 1) Cria anel perimetral e conector centroide->perímetro
    for i, row in quadras.iterrows():
        qid = str(row.quadra_id) if "quadra_id" in row else str(i)
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        # Anel externo (suporta MultiPolygon)
        ring = _outer_ring_line(geom)
        if ring is None or ring.is_empty or not isinstance(ring, LineString):
            # pula geometrias impossíveis
            continue

        perim_ls[qid] = ring

        # Densifica perímetro em pontos
        pts = _densify_line(ring, perim_step_m)
        perim_pts[qid] = pts

        ids = []
        for k, p in enumerate(pts):
            nid = f"{qid}:p{k}"
            ids.append(nid)
            G.add_node(nid, geometry=p, type="perimeter", qid=qid)
        perim_nodes[qid] = ids

        # Liga o anel (arestas consecutivas + fechamento)
        def _add_edge(u: str, v: str, geom: LineString):
            L = float(geom.length)
            weight_min = L / max(CFG.WALK_SPEED, 1e-6)  # minutos
            G.add_edge(
                u, v,
                geometry=geom,
                length_m=L,
                weight=weight_min,
                w=weight_min * 60.0
            )

        for k in range(len(ids) - 1):
            u, v = ids[k], ids[k + 1]
            _add_edge(u, v, LineString([G.nodes[u]["geometry"], G.nodes[v]["geometry"]]))
        if len(ids) >= 2:
            u, v = ids[-1], ids[0]
            _add_edge(u, v, LineString([G.nodes[u]["geometry"], G.nodes[v]["geometry"]]))

        # Nó do centroide (prioriza centroide.gpkg; fallback centroid geométrico)
        if qid in cent_map and cent_map[qid] is not None:
            cpt = cent_map[qid]
        else:
            try:
                cpt = _main_polygon(geom).centroid  # centroid do maior polígono
            except Exception:
                cpt = geom.centroid

        cid = f"{qid}:cent"
        elev = float(row["elevacao"]) if "elevacao" in row else 0.0
        G.add_node(cid, geometry=cpt, elev=elev, type="centroid", qid=qid)

        # Conector centroide -> nó de perímetro mais próximo
        if len(pts) > 0:
            k_near = _nearest_idx(cpt, pts)
            if k_near >= 0:
                pid = ids[k_near]
                seg = LineString([cpt, G.nodes[pid]["geometry"]])
                Lc = float(seg.length)
                w_min_c = Lc / max(CFG.WALK_SPEED, 1e-6)
                G.add_edge(
                    cid, pid,
                    geometry=seg,
                    length_m=Lc,
                    weight=w_min_c,
                    w=w_min_c * 60.0
                )

    # 2) Travessias entre perímetros de quadras vizinhas (mesmo sem encostar)
    for i, row in quadras.iterrows():
        qi = str(row.quadra_id) if "quadra_id" in row else str(i)
        ring_i = perim_ls.get(qi)
        if ring_i is None:
            continue

        # vizinhos por envelope do buffer (rápido)
        try:
            env = row.geometry.buffer(neighbor_search_buffer_m).envelope
            cand_idx = sidx.query(env)
        except Exception:
            # fallback: busca todos (mais lento, mas robusto)
            cand_idx = range(len(quadras))

        for j in cand_idx:
            if j <= i:
                continue
            qj = str(quadras.iloc[j].quadra_id) if "quadra_id" in quadras.columns else str(j)
            ring_j = perim_ls.get(qj)
            if ring_j is None:
                continue

            # Distância mínima entre anéis
            pi, pj = nearest_points(ring_i, ring_j)
            gap = float(pi.distance(pj))
            if gap > max_cross_gap_m:
                continue

            cross_line = LineString([pi, pj])
            # Evita travessias que cruzem barreiras "grandes"
            if cross_big_barrier(cross_line, bar_tree, bar_series):
                continue

            # Liga nós de perímetro mais próximos em cada anel
            list_i = perim_pts[qi]; ids_i = perim_nodes[qi]
            list_j = perim_pts[qj]; ids_j = perim_nodes[qj]
            ki = _nearest_idx(pi, list_i)
            kj = _nearest_idx(pj, list_j)
            if ki < 0 or kj < 0:
                continue
            ui = ids_i[ki]; vj = ids_j[kj]

            Lx = float(cross_line.length)
            w_min_x = Lx / max(CFG.WALK_SPEED, 1e-6)
            G.add_edge(
                ui, vj,
                geometry=cross_line,
                length_m=Lx,
                weight=w_min_x,
                w=w_min_x * 60.0
            )

    logger.info("grafo_perimetral: %d nós, %d arestas", G.number_of_nodes(), G.number_of_edges())
    return G

# ------------------------- Export -------------------------
def export_graph_to_files(G: nx.Graph, nodes_path: Path, edges_path: Path, crs: str):
    """Exporta nós e arestas para GPKG (camadas 'nodes' e 'edges')."""
    # NÓS
    nrows = []
    for n, d in G.nodes(data=True):
        geom = d.get("geometry")
        if geom is None:
            continue
        nrows.append({
            "node_id": str(n),
            "qid": d.get("qid"),
            "type": d.get("type"),
            "elev": d.get("elev", None),
            "geometry": geom
        })
    gdf_nodes = gpd.GeoDataFrame(nrows, geometry="geometry", crs=crs)
    gdf_nodes.to_file(nodes_path, driver="GPKG", layer="nodes", mode="w")

    # ARESTAS
    erows = []
    for u, v, d in G.edges(data=True):
        geom = d.get("geometry")
        if geom is None:
            # fallback: reta entre nós
            pu = G.nodes[u].get("geometry"); pv = G.nodes[v].get("geometry")
            if pu is not None and pv is not None:
                geom = LineString([pu, pv])
        if geom is None:
            continue
        erows.append({
            "u": str(u),
            "v": str(v),
            "length_m": float(d.get("length_m", geom.length)),
            "weight_min": float(d.get("weight", d.get("w", 0.0)/60.0 if d.get("w") else 0.0)),
            "w": float(d.get("w", d.get("weight", 0.0)*60.0)),
            "geometry": geom,
        })
    gdf_edges = gpd.GeoDataFrame(erows, geometry="geometry", crs=crs)
    gdf_edges.to_file(edges_path, driver="GPKG", layer="edges", mode="w")

# ------------------------- Carregamento -------------------------
def load_layers(
    quadras_path: Path,
    centroides_path: Path,
    barreiras_path: Path,
    *,
    crs: str
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame, gpd.GeoSeries | gpd.GeoDataFrame]:
    # QUADRAS
    quadras = gpd.read_file(quadras_path).to_crs(crs)
    if "quadra_id" not in quadras.columns:
        quadras["quadra_id"] = quadras.index.astype(str)

    # CENTROIDES
    centroides = gpd.read_file(centroides_path).to_crs(crs)

    # Garante coluna quadra_id nos centroides (sjoin_nearest se precisar)
    if "quadra_id" not in centroides.columns:
        logger.warning("centroide.gpkg sem 'quadra_id'; associando via sjoin_nearest...")
        qaux = quadras[["quadra_id", "geometry"]].rename(columns={"geometry": "qgeom"}).set_geometry("qgeom")
        c2 = gpd.sjoin_nearest(centroides, qaux, how="left")
        qcol = "quadra_id_right" if "quadra_id_right" in c2.columns else "quadra_id"
        centroides = gpd.GeoDataFrame(
            {"quadra_id": c2[qcol].astype(str), "geometry": c2.geometry},
            geometry="geometry", crs=crs
        )

    # Elevação (opcional)
    elev_col = next((c for c in ["elev", "elevacao", "z", "alt"] if c in centroides.columns), None)
    if elev_col is None:
        centroides["elev"] = 0.0
    else:
        centroides["elev"] = centroides[elev_col].astype(float)

    # BARREIRAS
    barreiras = gpd.read_file(barreiras_path).to_crs(crs)
    return quadras, centroides, barreiras

# ------------------------- CLI -------------------------
def _parse_args(argv=None):
    p = argparse.ArgumentParser(description="Constrói grafo perimetral com conectores e travessias")
    p.add_argument("--dir", type=Path, default=CFG.dataset_dir, help="Diretório base dos dados")
    p.add_argument("--quadras", type=Path, default=CFG.quadras_file, help="Arquivo das quadras (GPKG)")
    p.add_argument("--centroides", type=Path, default=CFG.centroides_file, help="Arquivo dos centroides (GPKG)")
    p.add_argument("--barreiras", type=Path, default=CFG.barreiras_file, help="Arquivo das barreiras (GPKG)")
    p.add_argument("--nos-out", type=Path, default=CFG.nos_out, help="Saída dos nós (GPKG)")
    p.add_argument("--arestas-out", type=Path, default=CFG.arestas_out, help="Saída das arestas (GPKG)")
    p.add_argument("--perim-step", type=float, default=CFG.PERIM_STEP_M, help="Passo de densificação do perímetro (m)")
    p.add_argument("--max-cross-gap", type=float, default=CFG.MAX_CROSS_GAP_M, help="Vão máximo para travessia (m)")
    p.add_argument("--neighbor-buffer", type=float, default=CFG.NEIGHBOR_SEARCH_BUFFER_M, help="Buffer para buscar vizinhos (m)")

    # Em notebooks, o ipykernel injeta "-f <json>"; vamos ignorar desconhecidos
    if argv is None:
        argv = sys.argv[1:]
    args, _unknown = p.parse_known_args(argv)
    return args

def main(argv=None):
    args = _parse_args(argv)

    # Atualiza CFG se vieram caminhos relativos
    CFG.dataset_dir = args.dir.expanduser().resolve()

    quadras_fp   = _file(args.quadras)
    centroides_fp= _file(args.centroides)
    barreiras_fp = _file(args.barreiras)
    nos_out_fp   = _file(args.nos_out)
    arestas_out_fp = _file(args.arestas_out)

    logger.info("Lendo camadas...")
    quadras, centroides, barreiras = load_layers(
        quadras_fp, centroides_fp, barreiras_fp, crs=CFG.CRS
    )

    # Campo auxiliar: centroide geométrico (se precisar em outras análises)
    if "centroide" not in quadras.columns:
        quadras["centroide"] = quadras.geometry.centroid

    logger.info("Gerando grafo (perímetro + conectores + travessias)...")
    G = build_graph_perimeter_rook_with_crossings(
        quadras=quadras,
        centroides_df=centroides,
        barreiras=barreiras,
        perim_step_m=float(args.perim_step),
        max_cross_gap_m=float(args.max_cross_gap),
        neighbor_search_buffer_m=float(args.neighbor_buffer),
    )

    logger.info("Exportando nós e arestas...")
    export_graph_to_files(G, nos_out_fp, arestas_out_fp, CFG.CRS)
    logger.info("Concluído: nós -> %s | arestas -> %s", nos_out_fp, arestas_out_fp)

if __name__ == "__main__":
    # Em notebook, chame sem argv (ignora '-f ...'); no CLI, usa sys.argv
    try:
        from IPython import get_ipython
        IN_IPY = get_ipython() is not None
    except Exception:
        IN_IPY = False

    if IN_IPY:
        main([])   # notebook
    else:
        main()     # CLI


INFO | 09:43:42 | grafo_builder | Lendo camadas...
INFO | 09:43:46 | grafo_builder | Gerando grafo (perímetro + conectores + travessias)...

KeyboardInterrupt



# Isócronas

In [11]:
# -*- coding: utf-8 -*-

from __future__ import annotations
from pathlib import Path
import os
import sys
import math
import time
import logging
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List, Set
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely.geometry import LineString, Point
from shapely.ops import unary_union, nearest_points
from shapely.strtree import STRtree

# =========================================================
# Config
# =========================================================

import re
import unicodedata

# Larguras aproximadas por classe viária (em metros)
# Só as > 25 m serão usadas como barreiras.
DEFAULT_CLASS_WIDTH_MAP = {
    "vtr": 60.0,         # vias de trânsito rápido
    "arterial": 45.0,
    "rodovia": 55.0,
    # classes menores ficam abaixo do limiar (25 m) e não viram barreiras
    "coletora": 22.0,
    "local": 18.0,
    "via de pedestre": 12.0,
}

def _normalize_class_value(s: str) -> str:
    """
    Normaliza rótulos de classe viária:
    - strip + lower
    - remove acentos
    - troca pontuação por espaço
    - colapsa espaços
    Ex.: ' VTR ' -> 'vtr'; 'Via de Pedestre' -> 'via de pedestre'
    """
    if s is None:
        return ""
    s = str(s).strip().lower()
    # remove acentos
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    # troca caracteres não alfanuméricos por espaço
    s = re.sub(r"[^\w\s]", " ", s)
    s = s.replace("_", " ")
    # colapsa espaços
    s = re.sub(r"\s+", " ", s).strip()
    return s


@dataclass
class Config:
    CRS: str = "EPSG:31983"

    # Orçamentos R1 (travado em 800 m)
    R1_M: float = 800.0                 # raio (m) métrico
    R1_T: float = 10.0                  # tempo (min)
    EPSILON: float = 0.0                # folga -> 0.0 para travar em 800 m (teste 2 retorno da folga em 1%)

    # Arquivos (nomes padrão; podem ser sobrescritos por --args)
    quadras_file: str = "quadras.gpkg"
    centroides_file: str = "centroide.gpkg"
    candidatos_file: str = "candidatos_hotspot.gpkg"   # << sementes quentes (1ª rodada)
    barreiras_file: str = "barreiras.gpkg"
    vias_file: str = "vias_1.gpkg"
    nos_file: str = "grafo_nos_queen.gpkg"
    arestas_file: str = "grafo_arestas_queen.gpkg"

    # Diretório base
    dataset_dir: Path = Path(
        os.environ.get(
            "DATA_DIR",
            "G:/.shortcut-targets-by-id/1lpu5GPlZuYcUIRMBBSrEQVYh5UH_MJ1v/Artigo-Isocronas/isocronas",
        )
    ).expanduser()

    # Parâmetros geométricos / rede
    WALK_SPEED: float = 70.0            # m/min
    ISO_VIA_BUF_M: float = 28.0         # buffer das vias no polígono
    ISO_POLY_GROW_M: float = 12.0       # grow-shrink p/ fechar fendas

    # Captura adicional por proximidade da rede
    CENTROID_CAPTURE_M: float = 110.0   # (m) distância do centroide até linhas alcançadas

    # Seed splitting vs barreiras (divide seed se perto da barreira p/ pegar ambos lados)
    BARRIER_NEAR_DIST: float = 50.0     # "perto" da barreira
    SPLIT_OFFSET_M: float = 200.0       # afastamento em lados opostos

    # Snap de seed ao grafo
    MAX_NODE_SNAP: float = 110.0        # (m) distância máx. seed -> nó

    # Precisão/agrupamento
    PREC: int = 2
    SEED_DEDUP_M: float = 100.0          # grade grossa p/ deduplicar seeds aproximadas (teste 2 duplicação por grade de 50 para 100)

    # Rodadas por ANEL (R2, R3, ...) – sempre a partir das seeds USADAS na rodada anterior
    R2_STEP_M: float = 1600.0           # raio do anel entre rodadas
    R2_RING_BAND_M: float = 150.0       # meia-largura do anel (seleciona d em [R-150, R+150])
    R2_RING_ADAPTIVE: bool = True       # amplia banda quando ninguém cai no anel
    R2_RING_BAND_GROW_M: float = 100.0  # passo de crescimento da banda adaptativa
    R2_RING_BAND_MAX_M: float = 600.0   # teto para a banda adaptativa

    # Seleção no anel
    R2_SECTORS: int = 16                 # 360°/setores (ex.: 8 -> 45°)  (teste 2 - alteração para cardinalidades intermediarias (SSE)
    R2_PER_SEED_TARGET: int = 0         # até K candidatos por seed anterior  (teste 2 - sem regras de candidatos por seed)
    R2_GLOBAL_CAP_FACTOR: float = 1.0   # cap global = K * n_seeds_prev * fator
    R2_MIN_SPACING_M: float = 1600.0     # afastamento mínimo entre candidatos finais (teste 2 - aumento de 800 para 1600)

    # Regras de qualidade
    MIN_QUADRAS_POR_ISO: int = 0       # descarta isócrona com menos que isso (teste 2 - exclusão da quadra minima)

    # Obrigatoriedade da 1ª rodada
    FIRST_ROUND_REQUIRES_CANDIDATES: bool = True

CFG = Config()

LOG_FMT = "%(levelname)s | %(asctime)s | %(name)s | %(message)s"
logging.basicConfig(level=logging.INFO, format=LOG_FMT, datefmt="%H:%M:%S")
logger = logging.getLogger("iso_class_net")

def _now(): return time.strftime("%H:%M:%S")
def log_info(msg: str, tag: str = "iso_class_net"):
    print(f"INFO | {_now()} | {tag} | {msg}")

# =========================================================
# Utilidades geométricas / barreiras
# =========================================================

def log_iso_seed(
    logger,
    iso_name,
    centroid_x,
    centroid_y,
    dist_m_array,
    n_total,
    n_new,
):
    """
    Gera a linha de log por isócrona com contagem de quadras totais, novas e antigas.
    """
    if n_total == 0:
        logger.info(
            f"seed {iso_name}: ({centroid_x:.2f}, {centroid_y:.2f}) -> 0 quadras | "
            f"raio[min=0, med=0, max=0] m"
        )
        return

    d = np.asarray(dist_m_array, dtype=float)
    rmin = float(np.nanmin(d))
    rmed = float(np.nanmedian(d))
    rmax = float(np.nanmax(d))
    n_old = n_total - n_new

    logger.info(
        f"seed {iso_name}: ({centroid_x:.2f}, {centroid_y:.2f}) -> {n_total} quadras "
        f"[novas={n_new}, antigas={n_old}] | "
        f"raio[min={rmin:.0f}, med={rmed:.0f}, max={rmax:.0f}] m"
    )

def clean_barriers(barreiras: gpd.GeoSeries, *, simplify_tol: float = 0.1) -> gpd.GeoSeries:
    g = barreiras.copy()
    g = g.buffer(0)
    if simplify_tol > 0:
        g = g.simplify(simplify_tol)
    return g

def load_vias_as_barriers(
    vias_path: Path,
    crs: str,
    *,
    class_field: str = "cvc_classe",
    min_width_m: float = 25.0,
    class_width_map: Optional[Dict[str, float]] = None,
) -> List:
    """
    Lê vias_1.gpkg e devolve UMA LISTA de polígonos-barreira:
    - Usa campo de classe (ex.: 'cvc_classe');
    - Mapeia classe -> largura (m) via class_width_map;
    - Mantém apenas classes com largura >= min_width_m (VTR, arterial, rodovia);
    - Faz buffer da linha por largura/2 e faz uma limpeza geométrica leve.
    """
    if class_width_map is None:
        class_width_map = DEFAULT_CLASS_WIDTH_MAP

    if not vias_path or not vias_path.exists():
        return []

    gdf = gpd.read_file(vias_path)
    if gdf.empty:
        return []

    gdf = gdf.to_crs(crs)

    if class_field not in gdf.columns:
        # sem campo de classe, não dá pra filtrar por VTR/arterial/rodovia
        return []

    # normaliza classes e converte em larguras
    classes_norm = gdf[class_field].apply(_normalize_class_value)
    widths = classes_norm.map(lambda v: float(class_width_map.get(v, 0.0)))

    mask = widths >= float(min_width_m)
    if not mask.any():
        return []

    sel = gdf[mask].copy()
    sel["__w__"] = widths[mask]

    bufs = []
    for geom, w in zip(sel.geometry.values, sel["__w__"].values):
        if geom is None or geom.is_empty:
            continue
        try:
            # largura efetiva ~ w => buffer = w/2
            bufs.append(geom.buffer(float(w) / 2.0))
        except Exception:
            continue

    if not bufs:
        return []

    # reaproveita sua função de limpeza para garantir polígonos válidos
    gs = gpd.GeoSeries(bufs, crs=crs)
    gs_clean = clean_barriers(gs, simplify_tol=0.5)
    return [g for g in gs_clean if g is not None and not g.is_empty]


def build_barrier_tree_with_vias(
    *,
    barreiras_path: Optional[Path],
    vias_path: Optional[Path],
    crs: str,
    class_field: str = "cvc_classe",
    min_width_m: float = 25.0,
    class_width_map: Optional[Dict[str, float]] = None,
) -> Optional[STRtree]:
    """
    Constrói uma STRtree única combinando:
      - barreiras.gpkg (como já era feito)
      - vias_1.gpkg, mas SOMENTE classes com largura >= min_width_m
        (VTR, arterial, rodovia, etc. conforme class_width_map).

    Retorna:
      - STRtree(geoms) ou None se nada for encontrado.
    """
    geoms_all: List = []

    # 1) Barreiras "tradicionais" (muros, rios, etc.)
    if barreiras_path and barreiras_path.exists():
        b = gpd.read_file(barreiras_path).to_crs(crs)
        if not b.empty:
            geoms_all.extend(list(clean_barriers(b.geometry, simplify_tol=0.5)))

    # 2) Vias largas como barreiras (VTR / arterial / rodovia)
    if vias_path and vias_path.exists():
        via_geoms = load_vias_as_barriers(
            vias_path=vias_path,
            crs=crs,
            class_field=class_field,
            min_width_m=min_width_m,
            class_width_map=class_width_map,
        )
        geoms_all.extend(via_geoms)

    if not geoms_all:
        return None

    return STRtree(geoms_all)

def process_iso_components_for_seed(
    seed_base_name,
    seed_xy,
    components,
    quadras_gdf,
    classified_mask,
    logger,
):
    """
    Atualiza classificação de quadras para uma seed e seus splits, logando novas/antigas.

    Parâmetros
    ----------
    seed_base_name : str
        Nome base da seed (ex: 'iso123').
        Se houver splits, serão criadas iso123a, iso123b, etc.
    seed_xy : (float, float)
        Coordenadas X, Y do centro da seed (para log).
    components : list[dict]
        Cada elemento deve ter:
            - 'quad_idx': np.ndarray de índices de quadras
            - 'dist_m':   np.ndarray de distâncias em metros
    quadras_gdf : GeoDataFrame
        GeoDataFrame das quadras com coluna 'iso_id' (ou similar) a ser preenchida.
    classified_mask : np.ndarray[bool]
        Máscara global de quadras já classificadas.
    logger : logging.Logger
        Logger principal (ex: logging.getLogger("iso_class_net"))

    Retorna
    -------
    n_new_total : int
        Número de novas quadras classificadas por esta seed (somando todos os splits).
    n_splits : int
        Número de splits gerados (len(components) - 1, se > 0).
    """
    seed_x, seed_y = seed_xy
    n_new_total = 0
    n_splits = max(0, len(components) - 1)

    # Gera sufixos para splits: '', 'a', 'b', 'c', ...
    suffixes = [""] + [chr(ord("a") + i) for i in range(max(0, len(components) - 1))]

    for comp_idx, comp in enumerate(components):
        idx_array = np.asarray(comp["quad_idx"], dtype=int)
        dist_m_array = np.asarray(comp["dist_m"], dtype=float)

        if idx_array.size == 0:
            continue

        # Quadras novas vs antigas
        comp_new_mask = ~classified_mask[idx_array]
        n_total = int(idx_array.size)
        n_new = int(comp_new_mask.sum())

        # Atualiza mask global e iso_id apenas para quadras novas
        if n_new > 0:
            classified_mask[idx_array[comp_new_mask]] = True

            # TODO: se sua coluna não se chama 'iso_id', troque aqui.
            iso_name = f"{seed_base_name}{suffixes[comp_idx]}"
            quadras_gdf.loc[idx_array[comp_new_mask], "iso_id"] = iso_name

        # Log de cada componente
        iso_name_log = f"{seed_base_name}{suffixes[comp_idx]}"
        log_iso_seed(
            logger=logger,
            iso_name=iso_name_log,
            centroid_x=seed_x,
            centroid_y=seed_y,
            dist_m_array=dist_m_array,
            n_total=n_total,
            n_new=n_new,
        )

        n_new_total += n_new

    return n_new_total, n_splits



def recommend_barrier_params_from_vias(
    *,
    min_width_m: float = 25.0,
    fallback_min_dist: float = 25.0,
) -> None:
    """
    Ajuste simples de CFG.BARRIER_NEAR_DIST usando o limiar de largura:
    near_dist ≈ min_width_m/2 + 10  (com piso em fallback_min_dist).
    """
    via_min = float(min_width_m)
    CFG.BARRIER_NEAR_DIST = max(float(fallback_min_dist), via_min / 2.0 + 10.0)
    log_info(f"BARRIER_NEAR_DIST ajustado automaticamente para {CFG.BARRIER_NEAR_DIST:.1f} m (base min_width={via_min} m)")


def split_seed_near_barrier(
    seed_pt: Point,
    tree: Optional[STRtree],
    *,
    near_dist: float | None = None,
    offset: float | None = None
) -> List[Point]:
    """
    Se a seed estiver a até 'near_dist' da barreira, gera dois pontos deslocados
    ('offset') em lados opostos; caso contrário, retorna a própria seed.
    Robusto a STRtree (Shapely 2.x) que pode retornar índices.
    """
    import numpy as np

    if near_dist is None: near_dist = CFG.BARRIER_NEAR_DIST
    if offset    is None: offset    = CFG.SPLIT_OFFSET_M
    if tree is None:
        return [seed_pt]

    try:
        hits = tree.query(seed_pt.buffer(max(near_dist, 1.0)), predicate="intersects")
    except Exception:
        return [seed_pt]

    # Normaliza 'hits' -> lista de geometrias
    cands: List = []
    geoms_seq = getattr(tree, "geometries", None)

    if isinstance(hits, np.ndarray):
        if geoms_seq is not None:
            for i in hits.tolist():
                try:
                    cands.append(geoms_seq[int(i)])
                except Exception:
                    pass
    elif isinstance(hits, (list, tuple)):
        cands = list(hits)
    else:
        try:
            if not hits:
                return [seed_pt]
        except Exception:
            return [seed_pt]

    # Seleciona a barreira mais próxima
    min_d = float("inf"); nearest_geom = None
    for cand in cands:
        geom = getattr(cand, "geometry", cand)
        if geom is None or getattr(geom, "is_empty", False):
            continue
        try:
            d = seed_pt.distance(geom)
        except Exception:
            continue
        if d < min_d:
            min_d, nearest_geom = d, geom

    if nearest_geom is None or min_d > near_dist:
        return [seed_pt]

    # Cria dois pontos em lados opostos à barreira
    p_bar = nearest_points(seed_pt, nearest_geom)[1]
    vx = seed_pt.x - p_bar.x; vy = seed_pt.y - p_bar.y
    norm = math.hypot(vx, vy)
    if norm < 1e-6:
        vx, vy, norm = 1.0, 0.0, 1.0
    ux, uy = vx / norm, vy / norm
    p1 = Point(p_bar.x + ux * offset, p_bar.y + uy * offset)
    p2 = Point(p_bar.x - ux * offset, p_bar.y - uy * offset)
    return [p1, p2]

# =========================================================
# Grafo: carregar nós/arestas já existentes
# =========================================================

def _pick_col(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

def load_graph_from_files(nodes_path: Path, edges_path: Path, *, crs: str) -> nx.Graph:
    if not nodes_path.exists():
        raise FileNotFoundError(f"Arquivo de nós não encontrado: {nodes_path}")
    if not edges_path.exists():
        raise FileNotFoundError(f"Arquivo de arestas não encontrado: {edges_path}")

    gdf_nodes = gpd.read_file(nodes_path).to_crs(crs)
    gdf_edges = gpd.read_file(edges_path).to_crs(crs)

    node_id_col = _pick_col(gdf_nodes.columns, ["node_id", "id", "osmid", "n", "node", "quadra_id"]) or "node_id"
    if node_id_col not in gdf_nodes.columns:
        gdf_nodes[node_id_col] = gdf_nodes.index.astype(str)

    elev_col = _pick_col(gdf_nodes.columns, ["elev", "elevation", "z", "alt", "altitude", "elevacao"]) or None

    ucol = _pick_col(gdf_edges.columns, ["u", "source", "from", "orig", "origem", "de"])
    vcol = _pick_col(gdf_edges.columns, ["v", "target", "to", "dest", "destino", "para"])
    if ucol is None or vcol is None:
        raise ValueError("Arestas sem colunas u/v (source/target).")

    length_col = _pick_col(gdf_edges.columns, ["length_m", "length", "len", "dist", "distance", "comprimento", "extensao"]) or "length_m"
    if length_col not in gdf_edges.columns:
        gdf_edges[length_col] = gdf_edges.geometry.length

    weight_col = _pick_col(gdf_edges.columns, ["weight", "tempo", "time", "cost", "custo", "weight_min"]) or None
    w_sec_col  = _pick_col(gdf_edges.columns, ["w", "weight_s", "tempo_s"]) or None
    if weight_col is None:
        if w_sec_col is not None:
            gdf_edges["weight"] = gdf_edges[w_sec_col].astype(float) / 60.0
        else:
            gdf_edges["weight"] = gdf_edges[length_col].astype(float) / CFG.WALK_SPEED
        weight_col = "weight"

    G = nx.Graph()
    for r in gdf_nodes.itertuples(index=False):
        attrs = {"geometry": getattr(r, "geometry")}
        if elev_col and hasattr(r, elev_col):
            try:
                attrs["elev"] = float(getattr(r, elev_col))
            except Exception:
                attrs["elev"] = 0.0
        G.add_node(str(getattr(r, node_id_col)), **attrs)

    for r in gdf_edges.itertuples(index=False):
        u = str(getattr(r, ucol)); v = str(getattr(r, vcol))
        if u == v:
            continue
        geom = getattr(r, "geometry")
        L_m  = float(getattr(r, length_col)) if length_col in gdf_edges.columns else (geom.length if geom is not None else 0.0)
        w_min = float(getattr(r, weight_col))
        attrs = dict(weight=w_min, length_m=L_m, geometry=geom)
        if w_sec_col and hasattr(r, w_sec_col):
            try: attrs["w"] = float(getattr(r, w_sec_col))
            except Exception: pass
        G.add_edge(u, v, **attrs)

    logger.info("grafo: %d nós, %d arestas", G.number_of_nodes(), G.number_of_edges())
    return G

# =========================================================
# Camadas de entrada e escrita de saída
# =========================================================

def _file(name: str | Path) -> Path:
    p = Path(name).expanduser()
    return p if p.is_absolute() else (CFG.dataset_dir / p)

def load_layers(
    quadras_path: Path,
    centroides_path: Path,
    candidatos_path: Optional[Path],
    barreiras_path: Optional[Path],
    *,
    crs: str
) -> Tuple[gpd.GeoDataFrame, gpd.GeoDataFrame, gpd.GeoDataFrame, Optional[STRtree]]:
    log_info("Lendo camadas...")

    quadras = gpd.read_file(quadras_path).to_crs(crs)
    if "quadra_id" not in quadras.columns:
        quadras["quadra_id"] = quadras.index.astype(str)
    if "elevacao" not in quadras.columns:
        quadras["elevacao"] = 0.0
    quadras["centroide"] = quadras.geometry.centroid

    centroides = gpd.read_file(centroides_path).to_crs(crs)
    if "quadra_id" not in centroides.columns:
        c2 = gpd.sjoin_nearest(
            centroides,
            quadras[["quadra_id", "geometry"]].rename(columns={"geometry": "qgeom"}).set_geometry("qgeom"),
            how="left"
        ).rename(columns={"quadra_id_right": "quadra_id"})
        centroides = gpd.GeoDataFrame(c2[["quadra_id", "geometry"]], geometry="geometry", crs=crs)

    # candidatos (seeds quentes primeiramente) — 1ª rodada exige este arquivo
    if candidatos_path and candidatos_path.exists():
        candidatos = gpd.read_file(candidatos_path).to_crs(crs)
    else:
        candidatos = gpd.GeoDataFrame(geometry=[], crs=crs)

    # barreiras -> STRtree robusto
    bar_tree = None
    if barreiras_path and barreiras_path.exists():
        b = gpd.read_file(barreiras_path).to_crs(crs)
        gclean = clean_barriers(b.geometry)
        bar_tree = STRtree(list(gclean))

    return quadras, centroides, candidatos, bar_tree

def write_layers(quadras: gpd.GeoDataFrame, *, out_gpkg: Path) -> None:
    out_gpkg = Path(out_gpkg)
    gdf_poly = gpd.GeoDataFrame(quadras.drop(columns=["centroide"]), geometry="geometry", crs=quadras.crs)
    gdf_poly.to_file(out_gpkg, layer="poligono", driver="GPKG", mode="w")
    gdf_cent = gpd.GeoDataFrame(quadras.drop(columns=["geometry"]), geometry="centroide", crs=quadras.crs)
    gdf_cent.to_file(out_gpkg, layer="centroide", driver="GPKG", mode="a")
    gdf_cent.drop(columns=["centroide"]).to_csv(out_gpkg.with_suffix(".csv"), index=False)

# =========================================================
# Auxiliares de rede / isócronas (sem bins/K)
# =========================================================

def sssp_func_attr(G: nx.Graph, attr: str):
    cache: Dict[str, Dict[str, float]] = {}
    def _f(src: str, cutoff: Optional[float] = None) -> Dict[str, float]:
        if src not in G: return {}
        if src not in cache:
            if attr == "length_m":
                wfun = lambda u, v, d: float(d.get("length_m", 0.0))
            elif attr == "weight":
                wfun = lambda u, v, d: float(d.get("weight", float(d.get("w", 0.0))/60.0))
            else:
                wfun = attr
            cache[src] = nx.single_source_dijkstra_path_length(G, src, weight=wfun)
        if cutoff is None:
            return cache[src]
        return {n: d for n, d in cache[src].items() if d <= cutoff}
    return _f

def iso_subgraph(G: nx.Graph, src_node: str, budget: float, weight: str):
    if weight == "length_m":
        wfun = lambda u, v, d: float(d.get("length_m", 0.0))
    elif weight == "weight":
        wfun = lambda u, v, d: float(d.get("weight", float(d.get("w", 0.0))/60.0))
    else:
        wfun = weight
    dist, _ = nx.single_source_dijkstra(G, src_node, weight=wfun, cutoff=budget)
    H = G.subgraph(dist.keys()).copy()
    nx.set_node_attributes(H, dist, name=f"dist_{weight}")
    return H, dist

def iso_subgraph_union(G: nx.Graph, src_node: str, *, t_budget_min: float, l_budget_m: float) -> nx.Graph:
    H = nx.Graph()
    Ht, dist_t = iso_subgraph(G, src_node, budget=t_budget_min, weight="weight")
    Hl, dist_l = iso_subgraph(G, src_node, budget=l_budget_m,  weight="length_m")
    nx.set_node_attributes(Ht, dist_t, name="dist_min")
    nx.set_node_attributes(Hl, dist_l, name="dist_m")
    H.add_nodes_from(Ht.nodes(data=True)); H.add_edges_from(Ht.edges(data=True))
    H.add_nodes_from(Hl.nodes(data=True)); H.add_edges_from(Hl.edges(data=True))
    return H

def iso_polygon_from_edges(H: nx.Graph, *, via_buf_m: float | None = None):
    if H is None or H.number_of_edges() == 0:
        return None
    if via_buf_m is None:
        via_buf_m = CFG.ISO_VIA_BUF_M

    geoms: List = []
    for _u, _v, d in H.edges(data=True):
        g = d.get("geometry")
        if g is not None and not g.is_empty:
            geoms.append(g)

    for u, v, d in H.edges(data=True):
        if d.get("geometry") is None:
            pu = H.nodes[u].get("geometry")
            pv = H.nodes[v].get("geometry")
            if pu is not None and pv is not None:
                geoms.append(LineString([pu, pv]))

    if not geoms:
        return None

    try:
        bufs = [g.buffer(via_buf_m) for g in geoms if g is not None and not g.is_empty]
        if not bufs:
            return None
        poly = unary_union(bufs)
        if poly is None or getattr(poly, "is_empty", True):
            return None

        grow = getattr(CFG, "ISO_POLY_GROW_M", 0.0) or 0.0
        if grow > 0:
            try:
                poly = poly.buffer(grow).buffer(-grow)
            except Exception:
                pass

        poly = poly.buffer(0)
        if getattr(poly, "is_empty", False):
            return None
        return poly
    except Exception:
        return None

def select_blocks_by_iso_polygon_no_bins(
    quadras: gpd.GeoDataFrame,
    iso_poly,
    *,
    centroid_capture_m: float,
    reachable_edges: Optional[List[LineString]] = None,
) -> gpd.GeoDataFrame:
    if iso_poly is None or quadras.empty:
        return quadras.iloc[0:0]

    inside = quadras[quadras.intersects(iso_poly)].copy()

    lines_union = None
    if reachable_edges:
        try:
            lines_union = unary_union([g for g in reachable_edges if g is not None and not g.is_empty])
        except Exception:
            lines_union = None

    if lines_union is not None:
        q2 = quadras[~quadras.index.isin(inside.index)].copy()
        if not q2.empty:
            q2["cent"] = q2.geometry.centroid
            q2["d_net"] = q2["cent"].apply(lambda c: c.distance(lines_union))
            near = q2[q2["d_net"] <= centroid_capture_m]
            if not near.empty:
                near = near.drop(columns=[c for c in ["cent", "d_net"] if c in near.columns])
                inside = pd.concat([inside, near], ignore_index=False)

    return inside

# =========================================================
# Classificação (rodada 1: candidatos; seguintes: ANEL de 1600 m)
# =========================================================

def attach_node_ids(G: nx.Graph, node_df: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    nodes = [(n, d) for n, d in G.nodes(data=True) if G.degree(n) > 0 and "geometry" in d]
    if not nodes:
        out = node_df.copy(); out["node_id"] = None; return out
    ids = [n for n, _ in nodes]
    pts = [d["geometry"] for _, d in nodes]
    kd = None
    try:
        from scipy.spatial import cKDTree
        kd = cKDTree(np.asarray([(p.x, p.y) for p in pts]))
    except Exception:
        kd = None

    out = node_df.copy()
    assigned = []
    for pt in out.geometry.values:
        if kd is None:
            arr = np.asarray([(p.x, p.y) for p in pts])
            d = np.hypot(arr[:,0] - pt.x, arr[:,1] - pt.y)
            idx = int(np.argmin(d))
        else:
            _, idx = kd.query([pt.x, pt.y], k=1)
        assigned.append(str(ids[int(idx)]))
    out["node_id"] = assigned
    hit = sum(1 for n in out["node_id"].values if n in G)
    logger.info("attach_node_ids: %d/%d (%.1f%%) quadras mapeadas p/ nós do grafo", hit, len(out), 100.0 * hit / max(1, len(out)))
    return out

def _dedup_points_grid(pts: List[Point], tol_m: float) -> List[Point]:
    if not pts: return []
    seen=set(); out=[]
    for p in pts:
        k = (int(round(p.x/tol_m)), int(round(p.y/tol_m)))
        if k in seen: continue
        seen.add(k); out.append(p)
    return out

def _los_blocked(seg: LineString, bar_tree: Optional[STRtree]) -> bool:
    if bar_tree is None or seg is None or seg.is_empty:
        return False
    try:
        hits = bar_tree.query(seg, predicate="intersects")
    except Exception:
        return False
    if hasattr(hits, "__len__"):
        return len(hits) > 0
    try:
        return bool(hits)
    except Exception:
        return False

import numpy as np
import logging

ring_debug = logging.getLogger("ring_debug")

def select_ring_seeds(
    quadras_gdf,
    classified_mask,
    prev_round_seed_coords,
    ring_inner_m,
    ring_outer_m,
    min_spacing_m,
    grid_size_m,
    max_seeds_per_round=None,
):
    """
    Seleciona seeds candidatas para a próxima rodada a partir do 'anel' de 1600 m.

    Parâmetros
    ----------
    quadras_gdf : GeoDataFrame
        GeoDataFrame das quadras (índice = id da quadra, geometry com polígonos).
    classified_mask : np.ndarray[bool]
        Booleano com True para quadras já classificadas (qualquer isócrona).
    prev_round_seed_coords : list[(x, y)]
        Lista de coordenadas (x,y) das seeds usadas na rodada anterior.
    ring_inner_m : float
        Raio interno do anel (ex: 800 m).
    ring_outer_m : float
        Raio externo do anel (ex: 1600 m).
    min_spacing_m : float
        Distância mínima entre seeds dentro da mesma rodada.
    grid_size_m : float
        Tamanho da célula de grade para espalhar sementes (ex: 800 ou 1000 m).
    max_seeds_per_round : int or None
        Limite opcional do número máximo de seeds por rodada.

    Retorna
    -------
    selected_indices : np.ndarray[int]
        Índices (do GeoDataFrame) das quadras escolhidas como seeds.
    metrics : dict
        Estatísticas para log: cand_total, in_ring, picked_by_sector, after_grid, after_spacing.
    """
    # -----------------------------
    # 1. Universo de candidatos = quadras ainda não classificadas
    # -----------------------------
    all_indices = quadras_gdf.index.to_numpy()
    mask_candidates = ~classified_mask  # quadras ainda livres
    cand_indices = all_indices[mask_candidates]
    cand_total = int(mask_candidates.sum())

    if len(prev_round_seed_coords) == 0 or cand_total == 0:
        # Sem seeds anteriores ou sem quadras livres -> nada a fazer
        ring_debug.info(
            "Anel r2: cand_total=%d | in_ring=0 | picked_by_sector=0 | "
            "after_grid=0 | after_spacing=0 | chosen=0",
            cand_total,
        )
        return np.array([], dtype=int), {
            "cand_total": cand_total,
            "in_ring": 0,
            "picked_by_sector": 0,
            "after_grid": 0,
            "after_spacing": 0,
            "chosen": 0,
        }

    # Coordenadas das quadras candidatas (centroide)
    cand_geom = quadras_gdf.geometry.values[mask_candidates]
    cand_x = np.array([g.centroid.x for g in cand_geom])
    cand_y = np.array([g.centroid.y for g in cand_geom])
    cand_coords = np.column_stack([cand_x, cand_y])  # (N, 2)

    # Coordenadas das seeds da rodada anterior
    prev_coords = np.array(prev_round_seed_coords, dtype=float)  # (M, 2)

    # -----------------------------
    # 2. Distância mínima até qualquer seed da rodada anterior
    # -----------------------------
    # Broadcasting: (N,1,2) - (1,M,2) -> (N,M,2)
    diff = cand_coords[:, None, :] - prev_coords[None, :, :]
    dist = np.sqrt((diff ** 2).sum(axis=2))  # (N, M)
    min_dist = dist.min(axis=1)  # (N,)

    # -----------------------------
    # 3. Filtro de anel: apenas cand dentro de [ring_inner_m, ring_outer_m]
    # -----------------------------
    in_ring_mask = (min_dist >= ring_inner_m) & (min_dist <= ring_outer_m)
    ring_indices = cand_indices[in_ring_mask]
    ring_coords = cand_coords[in_ring_mask]
    in_ring = int(in_ring_mask.sum())

    if in_ring == 0:
        ring_debug.info(
            "Anel r2: cand_total=%d | in_ring=0 | picked_by_sector=0 | "
            "after_grid=0 | after_spacing=0 | chosen=0",
            cand_total,
        )
        return np.array([], dtype=int), {
            "cand_total": cand_total,
            "in_ring": 0,
            "picked_by_sector": 0,
            "after_grid": 0,
            "after_spacing": 0,
            "chosen": 0,
        }

    # -----------------------------
    # 4. Espalhar por grade (after_grid)
    #    Uma seed por célula de grid, escolhendo a mais distante do núcleo (prioridade = min_dist)
    # -----------------------------
    # dist_ring = min_dist[in_ring_mask] -> reutilizar prioridade do "anel"
    dist_ring = min_dist[in_ring_mask]
    gx = np.floor(ring_coords[:, 0] / grid_size_m).astype(int)
    gy = np.floor(ring_coords[:, 1] / grid_size_m).astype(int)
    grid_ids = np.stack([gx, gy], axis=1)

    # Seleciona uma candidata por célula: a de maior dist_ring
    grid_to_best_idx = {}
    for i in range(len(ring_indices)):
        cell = (grid_ids[i, 0], grid_ids[i, 1])
        d = dist_ring[i]
        if cell not in grid_to_best_idx:
            grid_to_best_idx[cell] = i
        else:
            # Mantém a mais distante (prioridade)
            if d > dist_ring[grid_to_best_idx[cell]]:
                grid_to_best_idx[cell] = i

    grid_indices_local = np.array(list(grid_to_best_idx.values()), dtype=int)
    grid_ring_indices = ring_indices[grid_indices_local]
    grid_ring_coords = ring_coords[grid_indices_local]
    picked_by_sector = in_ring
    after_grid = len(grid_ring_indices)

    # -----------------------------
    # 5. Espaçamento mínimo entre seeds desta rodada (after_spacing)
    #    Importante: só contra seeds DA RODADA, não contra seeds antigas.
    # -----------------------------
    chosen = []
    chosen_coords = []

    # Ordena por distância ao núcleo (maior primeiro) para privilegiar seeds mais "externas"
    order = np.argsort(-dist_ring[grid_indices_local])  # decrescente
    for idx_local in order:
        idx_global = grid_ring_indices[idx_local]
        coord = grid_ring_coords[idx_local]

        ok = True
        for c in chosen_coords:
            dx = coord[0] - c[0]
            dy = coord[1] - c[1]
            if (dx * dx + dy * dy) < (min_spacing_m * min_spacing_m):
                ok = False
                break

        if ok:
            chosen.append(idx_global)
            chosen_coords.append(coord.tolist())

        if max_seeds_per_round is not None and len(chosen) >= max_seeds_per_round:
            break

    chosen = np.array(chosen, dtype=int)
    after_spacing = len(grid_ring_indices)
    chosen_n = len(chosen)

    ring_debug.info(
        "Anel r2: cand_total=%d | in_ring=%d | picked_by_sector=%d | "
        "after_grid=%d | after_spacing=%d | chosen=%d",
        cand_total, in_ring, picked_by_sector, after_grid, after_spacing, chosen_n
    )

    metrics = {
        "cand_total": cand_total,
        "in_ring": in_ring,
        "picked_by_sector": picked_by_sector,
        "after_grid": after_grid,
        "after_spacing": after_spacing,
        "chosen": chosen_n,
    }
    return chosen, metrics


def _classify_round(
    seeds: List[Point],
    quadras: gpd.GeoDataFrame,
    node_df: gpd.GeoDataFrame,
    G: nx.Graph,
    barreiras_tree: Optional[STRtree],
    *,
    iso_idx_start: int,
    classified: Set[str],
) -> Tuple[int, Dict[str, float], Dict[str, str], List[Point]]:
    """
    Uma rodada de classificação, sem K/bins (pega todas as quadras dentro do polígono + fallback).
    Descarta isócronas com menos de MIN_QUADRAS_POR_ISO quadras.
    Agora o log separa quadras novas vs já classificadas.
    """
    iso_tempo: Dict[str, float] = {}
    iso_id_map: Dict[str, str] = {}
    used_seeds: List[Point] = []
    iso_idx = iso_idx_start

    # distâncias na rede (cacheadas)
    get_time = sssp_func_attr(G, "weight")     # minutos

    # Mapa: quadra_id -> node_id
    q_to_node = dict(zip(node_df["quadra_id"].astype(str), node_df["node_id"]))

    # KD para snap da seed ao grafo
    nodes = [(n, d) for n, d in G.nodes(data=True) if G.degree(n) > 0 and "geometry" in d]
    ids   = [n for n, _ in nodes]
    coords = np.asarray([(d["geometry"].x, d["geometry"].y) for _, d in nodes], dtype=float)
    kd = None
    try:
        from scipy.spatial import cKDTree
        kd = cKDTree(coords) if len(coords) else None
    except Exception:
        kd = None

    def _snap_seed_to_graph(seed_pt: Point) -> Optional[str]:
        if kd is None or len(ids) == 0:
            return None
        dist, idx = kd.query([seed_pt.x, seed_pt.y], k=1)
        if CFG.MAX_NODE_SNAP and float(dist) > float(CFG.MAX_NODE_SNAP):
            return None
        return str(ids[int(idx)])

    seeds_in = len(seeds)
    split_total = skipped_snap = skipped_poly = 0

    for seed in seeds:
        # split em função de barreira (mantém lógica que você já tinha)
        split_pts = split_seed_near_barrier(
            seed, barreiras_tree,
            near_dist=CFG.BARRIER_NEAR_DIST, offset=CFG.SPLIT_OFFSET_M
        )
        split_total += max(0, len(split_pts) - 1)

        for s in split_pts:
            src = _snap_seed_to_graph(s)
            if src is None:
                skipped_snap += 1
                continue

            # Orçamentos (travados)
            t_budget = CFG.R1_T * (1 + CFG.EPSILON)
            l_budget = CFG.R1_M * (1 + CFG.EPSILON)

            H = iso_subgraph_union(G, src_node=src, t_budget_min=t_budget, l_budget_m=l_budget)
            if H is None or H.number_of_edges() == 0:
                skipped_poly += 1
                continue

            # Polígono
            iso_poly = iso_polygon_from_edges(H, via_buf_m=CFG.ISO_VIA_BUF_M)
            if iso_poly is None or getattr(iso_poly, "is_empty", True):
                skipped_poly += 1
                continue

            # Linhas alcançadas
            reach_lines = [d.get("geometry") for _u, _v, d in H.edges(data=True)]

            # Seleciona TODAS as quadras dentro + fallback por proximidade à rede
            selecionadas = select_blocks_by_iso_polygon_no_bins(
                quadras, iso_poly,
                centroid_capture_m=CFG.CENTROID_CAPTURE_M,
                reachable_edges=reach_lines
            )

            # DESCARTE por tamanho mínimo
            if selecionadas.empty or (len(selecionadas) < CFG.MIN_QUADRAS_POR_ISO):
                skipped_poly += 1
                continue

            iso_id = f"iso{iso_idx}"
            iso_idx += 1
            used_seeds.append(s)

            # distâncias na rede
            dist_t = get_time(src, cutoff=t_budget)    # minutos

            # raio euclidiano (seed → centroides)
            centroids = selecionadas.geometry.centroid
            qids = selecionadas["quadra_id"].astype(str).values
            re = np.array([s.distance(c) for c in centroids.values], dtype=float) if len(centroids) else np.array([], dtype=float)

            # novas vs antigas
            if len(qids):
                is_new = np.array([qid not in classified for qid in qids], dtype=bool)
                n_total = len(qids)
                n_new = int(is_new.sum())
            else:
                is_new = np.array([], dtype=bool)
                n_total = 0
                n_new = 0

            # log detalhado (usa a função log_iso_seed que você já tem no arquivo)
            log_iso_seed(
                logger=logger,
                iso_name=iso_id,
                centroid_x=s.x,
                centroid_y=s.y,
                dist_m_array=re,
                n_total=n_total,
                n_new=n_new,
            )

            # atribuições APENAS para quadras novas
            for qid, cent, flag_new in zip(qids, centroids.values, is_new):
                if not flag_new:
                    continue

                node_id = q_to_node.get(qid)

                # tempo via rede (min) se possível; SENÃO fallback euclidiano seed->centroide
                t_min = None
                if node_id and node_id in dist_t:
                    t_min = float(dist_t[node_id])
                if t_min is None:
                    d_eucl = float(s.distance(cent))
                    t_min = d_eucl / CFG.WALK_SPEED

                iso_tempo[qid] = float(t_min)
                iso_id_map[qid] = iso_id
                classified.add(qid)

    logger.info(
        "rodada: seeds_in=%d; splits=%d; usadas=%d; novas_quadras=%d; descartadas[snap=%d, poligono=%d]",
        seeds_in, split_total, len(used_seeds), len(iso_tempo), skipped_snap, skipped_poly
    )

    return iso_idx, iso_tempo, iso_id_map, used_seeds

def pipeline_r1_r2(
    quadras: gpd.GeoDataFrame,
    node_df: gpd.GeoDataFrame,
    candidatos_gdf: gpd.GeoDataFrame,
    barreiras_tree: Optional[STRtree],
    *,
    G: nx.Graph,
    max_rounds: int = 999,              # número máximo de rodadas (alto; parada real é por min_new_quadras)
    min_new_quadras: int = 1,           # progresso mínimo por rodada para continuar
) -> gpd.GeoDataFrame:
    """
    Classificação sem bins/K, usando o grafo existente.

    Rodada 1:
        - seeds = candidatos (candidatos_hotspot.gpkg) EXCLUSIVAMENTE, se
          CFG.FIRST_ROUND_REQUIRES_CANDIDATES=True.

    Rodadas 2+:
        - seeds = centroides no ANEL (R2_STEP_M ± banda) das seeds USADAS na rodada anterior,
          com setores, LOS, dedupe em grid e afastamento mínimo.

    O laço termina quando:
        - Não há seeds para a rodada, ou
        - new_q < min_new_quadras (sem progresso), ou
        - r atinge max_rounds (limite de segurança).
    """
    # Garante mapeamento quadra_id -> node_id
    node_df = attach_node_ids(G, node_df)
    if "node_id" not in node_df.columns:
        raise RuntimeError("node_df sem 'node_id' após attach_node_ids().")

    classified: Set[str] = set()
    iso_tempo_all: Dict[str, float] = {}
    iso_id_all: Dict[str, str] = {}

    # Seeds - Rodada 1 DEVE usar candidatos; se vazio e flag ligada, aborta
    if (candidatos_gdf is None or candidatos_gdf.empty) and CFG.FIRST_ROUND_REQUIRES_CANDIDATES:
        raise RuntimeError("candidatos_hotspot.gpkg ausente/vazio, e FIRST_ROUND_REQUIRES_CANDIDATES=True.")
    seeds_round1 = list(candidatos_gdf.geometry)
    seeds_round1 = _dedup_points_grid([s for s in seeds_round1 if s is not None], CFG.SEED_DEDUP_M)

    # Prepara iterador de rodadas
    iso_idx = 1
    prev_total_q = 0
    used_round_prev: List[Point] = []

    R = max(1, int(max_rounds))
    for r in range(1, R + 1):
        if r == 1:
            seeds = seeds_round1
            origem = "candidatos"
        else:
            # -----------------------------
            # Rodadas 2+ : seeds no ANEL em torno das seeds usadas na rodada anterior
            # -----------------------------
            ring_band = CFG.R2_RING_BAND_M
            seeds = []

            if used_round_prev:
                # coords das seeds da rodada anterior
                prev_coords = [(p.x, p.y) for p in used_round_prev]

                # máscara de quadras já classificadas, para evitar virar seed
                if iso_id_all:
                    classified_mask = quadras["quadra_id"].astype(str).isin(iso_id_all.keys()).to_numpy()
                else:
                    classified_mask = np.zeros(len(quadras), dtype=bool)

                # teto global opcional de seeds
                max_seeds = None
                if CFG.R2_PER_SEED_TARGET > 0:
                    max_seeds = int(len(prev_coords) * CFG.R2_PER_SEED_TARGET * CFG.R2_GLOBAL_CAP_FACTOR)

                # tenta com banda base + banda adaptativa (se ligado)
                while True:
                    ring_inner = max(0.0, CFG.R2_STEP_M - ring_band)
                    ring_outer = CFG.R2_STEP_M + ring_band

                    sel_idx, _metrics = select_ring_seeds(
                        quadras_gdf=quadras,
                        classified_mask=classified_mask,
                        prev_round_seed_coords=prev_coords,
                        ring_inner_m=ring_inner,
                        ring_outer_m=ring_outer,
                        min_spacing_m=CFG.R2_MIN_SPACING_M,
                        grid_size_m=CFG.R2_STEP_M,
                        max_seeds_per_round=max_seeds,
                    )

                    if sel_idx.size > 0 or not CFG.R2_RING_ADAPTIVE or ring_band >= CFG.R2_RING_BAND_MAX_M:
                        break

                    ring_band += CFG.R2_RING_BAND_GROW_M

                if sel_idx.size > 0:
                    # usa o centroide das quadras selecionadas como nova seed
                    cent_series = quadras.loc[sel_idx, "centroide"]
                    seeds = [pt for pt in cent_series.values if pt is not None]

            origem = f"anel_{CFG.R2_STEP_M:.0f}m_de_round{r-1}"

        logger.info("Rodada %d (%s) - seeds: %d", r, origem, len(seeds))
        if not seeds:
            logger.info("Sem seeds para a rodada %d. Encerrando.", r)
            break

        # roda a classificação para esta rodada
        iso_idx, iso_tempo, iso_id_map, used_round = _classify_round(
            seeds, quadras, node_df, G, barreiras_tree,
            iso_idx_start=iso_idx, classified=classified,
        )
        used_round_prev = used_round  # alimenta o anel da próxima rodada

        # acumula resultados
        iso_tempo_all.update(iso_tempo)
        iso_id_all.update(iso_id_map)

        total_q = len(iso_tempo_all)
        new_q = total_q - prev_total_q
        logger.info("Progresso: novas_quadras=%d (acumulado=%d)", new_q, total_q)

        # critério de parada por falta de progresso
        if new_q < min_new_quadras and r < R:
            logger.info(
                "Parada antecipada: rodada %d sem progresso suficiente (min=%d).",
                r, min_new_quadras
            )
            break

        prev_total_q = total_q

    # aplica atributos de saída
    out = quadras.copy()
    out["iso_tempo"] = out.quadra_id.map(lambda q: iso_tempo_all.get(str(q)))
    out["iso_id"]    = out.quadra_id.map(lambda q: iso_id_all.get(str(q)))
    return out

# =========================================================
# CLI
# =========================================================

def _parse_args(argv=None):
    import argparse

    p = argparse.ArgumentParser(
        description="Classificação de isócronas (sem bins/K) usando grafo existente com seeds em anéis sucessivos."
    )

    # Caminhos principais
    p.add_argument("--dir", type=Path, default=CFG.dataset_dir, help="Diretório base dos dados.")
    p.add_argument("--quadras", type=Path, default=CFG.quadras_file, help="GPKG de quadras.")
    p.add_argument("--centroides", type=Path, default=CFG.centroides_file, help="GPKG de centroides (com quadra_id).")
    p.add_argument("--candidatos", type=Path, default=CFG.candidatos_file, help="GPKG de seeds quentes (candidatos_hotspot.gpkg).")
    p.add_argument("--barreiras", type=Path, default=CFG.barreiras_file, help="GPKG de barreiras (opcional).")
    p.add_argument("--nos", type=Path, default=CFG.nos_file, help="GPKG dos nós do grafo.")
    p.add_argument("--arestas", type=Path, default=CFG.arestas_file, help="GPKG das arestas do grafo.")
    p.add_argument("--out", type=Path, default=Path("isocronas_classificacao.gpkg"), help="Arquivo de saída (GPKG).")

    # Ajustes finos (opcionais)
    p.add_argument(
        "--max-rounds",
        type=int,
        default=999,
        help="Número máximo de rodadas (padrão=999; na prática o pipeline para quando não houver novas quadras).",
    )
    p.add_argument(
        "--seed-dedup-m",
        type=float,
        default=CFG.SEED_DEDUP_M,
        help="Grade de dedupe de seeds (m).",
    )
    p.add_argument(
        "--ring-band-m",
        type=float,
        default=CFG.R2_RING_BAND_M,
        help="Meia-largura do anel (m).",
    )
    p.add_argument(
        "--ring-band-max-m",
        type=float,
        default=CFG.R2_RING_BAND_MAX_M,
        help="Máximo da banda adaptativa (m).",
    )
    p.add_argument(
        "--ring-grow-m",
        type=float,
        default=CFG.R2_RING_BAND_GROW_M,
        help="Crescimento da banda adaptativa (m).",
    )
    p.add_argument(
        "--r2-sectors",
        type=int,
        default=CFG.R2_SECTORS,
        help="Nº de setores angulares por seed (padrão=8 ou o valor configurado em CFG).",
    )
    p.add_argument(
        "--r2-per-seed-target",
        type=int,
        default=CFG.R2_PER_SEED_TARGET,
        help="Máx. candidatos por seed no anel (0 = sem limite por seed).",
    )
    p.add_argument(
        "--r2-global-cap-factor",
        type=float,
        default=CFG.R2_GLOBAL_CAP_FACTOR,
        help="Fator do teto global de seeds (padrão=1.0).",
    )
    p.add_argument(
        "--r2-min-spacing-m",
        type=float,
        default=CFG.R2_MIN_SPACING_M,
        help="Afastamento mínimo entre candidatos (m).",
    )
    p.add_argument(
        "--min-quadras-iso",
        type=int,
        default=CFG.MIN_QUADRAS_POR_ISO,
        help="Mínimo de quadras por isócrona (padrão atual em CFG, ex.: 0 = sem filtro).",
    )
    p.add_argument(
        "--first-round-requires-candidates",
        action="store_true",
        default=CFG.FIRST_ROUND_REQUIRES_CANDIDATES,
        help="Se ligado (default), aborta se não houver candidatos_hotspot na 1ª rodada.",
    )

    # Aceita rodar em notebook sem quebrar por '-f ...'
    if argv is None:
        argv = sys.argv[1:]
    args, _unknown = p.parse_known_args(argv)
    return args


def main(argv=None):
    args = _parse_args(argv)

    # Atualiza base de configuração a partir dos argumentos
    CFG.dataset_dir = args.dir.expanduser().resolve()
    CFG.SEED_DEDUP_M = float(args.seed_dedup_m)
    CFG.R2_RING_BAND_M = float(args.ring_band_m)
    CFG.R2_RING_BAND_MAX_M = float(args.ring_band_max_m)
    CFG.R2_RING_BAND_GROW_M = float(args.ring_grow_m)
    CFG.R2_SECTORS = int(args.r2_sectors)
    CFG.R2_PER_SEED_TARGET = int(args.r2_per_seed_target)
    CFG.R2_GLOBAL_CAP_FACTOR = float(args.r2_global_cap_factor)
    CFG.R2_MIN_SPACING_M = float(args.r2_min_spacing_m)
    CFG.MIN_QUADRAS_POR_ISO = int(args.min_quadras_iso)
    CFG.FIRST_ROUND_REQUIRES_CANDIDATES = bool(args.first_round_requires_canditates) if hasattr(args, "first_round_requires_canditates") else bool(args.first_round_requires_candidates)

    # Resolve caminhos de arquivos
    quad_fp = _file(args.quadras)
    cent_fp = _file(args.centroides)
    cand_fp = _file(args.candidatos)
    bar_fp  = _file(args.barreiras)
    nos_fp  = _file(args.nos)
    are_fp  = _file(args.arestas)
    out_fp  = _file(args.out)

    log_info(f"Dataset dir: {CFG.dataset_dir}")
    log_info(f"Nos: {nos_fp}")
    log_info(f"Arestas: {are_fp}")

    # Carrega camadas principais (quadras, centroides, candidatos)
    quadras, centroides, candidatos, _ = load_layers(
        quadras_path=quad_fp,
        centroides_path=cent_fp,
        candidatos_path=cand_fp if cand_fp.exists() else None,
        barreiras_path=None,  # barreiras tratadas abaixo em conjunto com vias
        crs=CFG.CRS,
    )

    # --- Barreiras combinadas: barreiras.gpkg + vias_1.gpkg (VTR / arterial / rodovia > 25 m) ---
    vias_fp = _file("vias_1.gpkg")  # ou use CFG.vias_file se preferir parametrizar

    bar_tree = build_barrier_tree_with_vias(
        barreiras_path=bar_fp if bar_fp.exists() else None,
        vias_path=vias_fp if vias_fp.exists() else None,
        crs=CFG.CRS,
        class_field="cvc_classe",  # ajuste se o campo tiver outro nome
        min_width_m=25.0,
    )

    # Ajuste leve do raio de influência da barreira
    recommend_barrier_params_from_vias(min_width_m=25.0)

    # 1ª rodada: exige candidatos se flag ligada
    if (candidatos is None or candidatos.empty) and CFG.FIRST_ROUND_REQUIRES_CANDIDATES:
        raise RuntimeError("candidatos_hotspot.gpkg ausente/vazio — a 1ª rodada deve ser EXCLUSIVAMENTE com candidatos.")

    # Carrega grafo existente (nós + arestas)
    log_info("Carregando grafo (nós + arestas)...")
    G = load_graph_from_files(nos_fp, are_fp, crs=CFG.CRS)

    # Prepara node_df (centroides com elevação opcional)
    node_df = centroides.copy()
    elev_col = next((c for c in ["elev", "elevacao", "z", "alt"] if c in node_df.columns), None)
    node_df["elev"] = node_df[elev_col] if elev_col else 0.0

    # Seeds: 1ª rodada com candidatos; rodadas seguintes por ANEL
    log_info("Classificando pela rede (1ª rodada = candidatos; seguintes = anel 1600 m) ...")
    out = pipeline_r1_r2(
        quadras=quadras,
        node_df=node_df,
        candidatos_gdf=candidatos,
        barreiras_tree=bar_tree,
        G=G,
        max_rounds=int(args.max_rounds),
        min_new_quadras=1,
    )

    # Estatística simples por isócrona (tamanho do "bairro")
    sizes = out.groupby("iso_id", dropna=True).size().sort_values(ascending=False)
    if not sizes.empty:
        logger.info("Resumo - quadras por isócrona (top 10):\n%s", sizes.head(10).to_string())

    # Cobertura global: quantas quadras foram de fato classificadas
    n_total = len(out)
    n_class = out["iso_id"].notna().sum()
    perc = 100.0 * n_class / max(1, n_total)
    log_info(f"Cobertura: {n_class}/{n_total} quadras ({perc:.1f}%) classificadas")

    # Escrita de saída
    write_layers(out, out_gpkg=out_fp)
    log_info(f"Saída escrita em {out_fp}")

# ---------------------------------------------------------

if __name__ == "__main__":
    # Suporte a execução dentro de notebook
    try:
        get_ipython  # type: ignore
        IN_IPY = True
    except Exception:
        IN_IPY = False

    if IN_IPY:
        main([])   # notebook
    else:
        main()


INFO | 14:30:16 | iso_class_net | Dataset dir: G:\.shortcut-targets-by-id\1lpu5GPlZuYcUIRMBBSrEQVYh5UH_MJ1v\Artigo-Isocronas\isocronas
INFO | 14:30:16 | iso_class_net | Nos: G:\.shortcut-targets-by-id\1lpu5GPlZuYcUIRMBBSrEQVYh5UH_MJ1v\Artigo-Isocronas\isocronas\grafo_nos_queen.gpkg
INFO | 14:30:16 | iso_class_net | Arestas: G:\.shortcut-targets-by-id\1lpu5GPlZuYcUIRMBBSrEQVYh5UH_MJ1v\Artigo-Isocronas\isocronas\grafo_arestas_queen.gpkg
INFO | 14:30:16 | iso_class_net | Lendo camadas...
INFO | 14:30:28 | iso_class_net | BARRIER_NEAR_DIST ajustado automaticamente para 25.0 m (base min_width=25.0 m)
INFO | 14:30:28 | iso_class_net | Carregando grafo (nós + arestas)...


INFO | 14:31:03 | iso_class_net | grafo: 1156853 nós, 1382092 arestas


INFO | 14:31:03 | iso_class_net | Classificando pela rede (1ª rodada = candidatos; seguintes = anel 1600 m) ...


INFO | 14:31:26 | iso_class_net | attach_node_ids: 63836/63836 (100.0%) quadras mapeadas p/ nós do grafo
INFO | 14:31:26 | iso_class_net | Rodada 1 (candidatos) - seeds: 58
INFO | 14:31:52 | iso_class_net | seed iso1: (332302.10, 7396476.18) -> 99 quadras [novas=99, antigas=0] | raio[min=24, med=534, max=803] m
INFO | 14:31:54 | iso_class_net | seed iso2: (350907.98, 7395468.33) -> 7 quadras [novas=7, antigas=0] | raio[min=89, med=153, max=857] m
INFO | 14:31:57 | iso_class_net | seed iso3: (350507.98, 7395468.33) -> 47 quadras [novas=45, antigas=2] | raio[min=92, med=469, max=1133] m
INFO | 14:32:02 | iso_class_net | seed iso4: (326195.49, 7395044.18) -> 149 quadras [novas=149, antigas=0] | raio[min=34, med=527, max=839] m
INFO | 14:32:07 | iso_class_net | seed iso5: (332133.41, 7393747.04) -> 118 quadras [novas=118, antigas=0] | raio[min=84, med=515, max=834] m
INFO | 14:32:11 | iso_class_net | seed iso6: (321400.51, 7410260.75) -> 97 quadras [novas=97, antigas=0] | raio[min=71, med=

INFO | 16:28:41 | iso_class_net | Cobertura: 63187/63836 quadras (99.0%) classificadas


INFO | 16:28:45 | pyogrio._io | Created 63,836 records
INFO | 16:29:33 | pyogrio._io | Created 63,836 records


INFO | 16:29:34 | iso_class_net | Saída escrita em G:\.shortcut-targets-by-id\1lpu5GPlZuYcUIRMBBSrEQVYh5UH_MJ1v\Artigo-Isocronas\isocronas\isocronas_classificacao.gpkg
